
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 강의 - 단일 AI 에이전트 작성과 Databricks Mosaic AI 에이전트 Framework

이 강의 노트북은 Databricks가 포괄적인 Mosaic AI 에이전트 Framework를 통해 단일 AI 에이전트 개발을 어떻게 지원하는지 알아봅니다. 데이터브릭스 플랫폼에서 상용화가 가능한 AI 에이전트를 구축하기 위한 도구와 프레임워크, 그리고 최적의 실무 사례(best practices)를 살펴보겠습니다.

## 개요

Databricks Mosaic AI 에이전트 프레임워크는 에이전트 작성, 배포, AI 에이전트 모니터링을 위한 통합 플랫폼을 제공합니다. 이 프레임워크는 LangChain, LangGraph, DSPy, OpenAI 등 여러 인기 있는 에이전트 개발 라이브러리를 지원하며, AI search, Model Serving, MLflow 같은 Databricks 서비스와의 네이티브 통합도 제공합니다.

이 프레임워크는 자동 추적(tracing), 종합적인 평가 기능, 모자이크 AI 모델 서빙(Mosaic AI Model Serving)으로의 원활한 배포 등을 통해 실제 서비스 적용(production readiness)에 초점을 맞추고 있습니다. 단순한 챗봇 에이전트부터 복잡한 멀티 에이전트 시스템에 이르기까지, 데이터브릭스는 기업 규모의 AI 애플리케이션에 필요한 도구와 인프라를 제공합니다.

## 학습 목표

_이 강의를 마치면 여러분은 다음을 수행할 수 있게 됩니다._
- Databricks Mosaic AI 에이전트 프레임워크의 아키텍처와 구성 요소를 이해합니다.
- 상용 등급의 에이전트 개발에 ResponsesAgent를 사용할 때 얻을 수 있는 장점을 설명할 수 있습니다.
- 지원되는 에이전트 개발 프레임워크와 이들의 통합 패턴을 파악합니다.
- 에이전트 배포 시 고려해야 할 사항을 기술할 수 있습니다.
- 스트리밍, 맞춤형 입출력, 리트리버(retriever) 연동 등과 같은 고급 기능을 인지하고 활용합니다.

## A. Mosaic AI 에이전트 프레임워크 소개

Databricks Mosaic AI 에이전트 Framework는 엔터프라이즈 규모에서 AI 에이전트를 구축, 배포, 관리하기 위한 종합 솔루션입니다. 이 Framework는 개발부터 생산까지 전체 _에이전트 수명주기_ 모니터링을 다룹니다.

### 에이전트의 생애 주기
에이전트의 생애주기는 다음과 같이 요약할 수 있습니다:
1. **데이터를 준비하고 도구를 만들기**:  
    - 이 단계에서는 Notebook, SQL 쿼리, Lakeflow 제품군을 사용하여 AI 관련 ETL을 수행합니다. 일반적으로 AI 엔지니어가 AI Search를 통해 비정형 데이터를 임베딩하고 인덱싱하는 단계가 여기에 해당합니다. 
    - 데이터 준비가 완료되면 엔지니어는 SQL 또는 Python 문법으로 도구를 생성하고, 통합 거버넌스를 위해 해당 도구를 [Unity Catalog](https://docs.databricks.com/aws/en/data-governance/unity-catalog/)에 등록합니다.
1. **품질 검증을 동반한 신속한 프로토타이핑**
    - 이 단계에서는 일반적으로 AI Playground의 노코드 인터페이스를 통해 빠른 프로토타이핑 에이전트를 위한 신속 테스트가 수행됩니다. 여기서 시스템 프롬프트를 지정하고, 다양한 모델을 선택해 나란히 비교하며, 결과의 직관적인 완성도(vibe check)를 확인할 수 있습니다.
    -  그 후 AI 엔지니어는 품질 문제와 그 근본 원인을 파악하도록 설계된 [MLflow 3의 평가 도구](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)를를 사용해 콘텐츠를 평가합니다.
    - 신속한 프로토타이핑이 완료되면 Playground에서 코드를 내보내 `mlflow.genai.evaluate()`를 활용할 수 있습니다([여기](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/eval-harness)서 더 읽어보기).  
1. **평가 및 피드백 수집**
    - LLM 심사위원(LLM judges), 관계자 라벨링, 합성 데이터 등의 방법을 사용하여 평가 데이터셋을 기준으로 에이전트를 테스트합니다. 대개 리뷰 앱이나 상호작용의 직접적인 추적(tracing)을 통해 이해관계자 및 도메인 전문가의 피드백을 수집합니다.
1. **데이터 및 피드백 라벨링**
    - 향후 에이전트 반복 개발 시 테스트에 사용할 고품질 벤치마크를 만들기 위해 상호작용과 출력값에 라벨을 지정합니다. 이를 통해 품질 평가의 기준(ground truth) 역할을 하는 평가 세트가 생성됩니다.
    - 라벨링 세션을 통해 도메인 전문가로부터 생성형 AI 애플리케이션의 동작에 대한 피드백을 체계적으로 수집할 수 있습니다.  라벨링 세션에 대해 더 읽어보시려면 [여기](https://learn.microsoft.com/ko-kr/azure/databricks/mlflow3/genai/human-feedback/concepts/labeling-sessions)에서 읽을 수 있습니다.
1. **반복적인 개선**
    - 피드백과 벤치마크 결과를 활용하여 품질 문제의 근본 원인을 파악하고 해결합니다.
    - 원하는 수준의 정확도, 안전성, 비용, 대기 시간(latency) 간의 균형을 맞추기 위해 여러 버전과 구성을 평가합니다.
1. **운영 환경 배포**
    - 에이전트가 개발 단계에서 확장 가능한 운영 환경(대개 Model Serving을 통한 REST API 형태)으로 전환됩니다. 이 단계에서 에이전트는 Unity Catalog와 같은 구성 요소를 활용해 통합 거버넌스를 구현함으로써 접근 권한과 규정 준수를 관리받습니다.
1. **품질 및 성능 모니터링**
    - 배포 후에도 에이전트는 개발 단계와 동일한 평가 및 추적 도구를 통해 지속적으로 모니터링됩니다. 로그, 추적, 사용자 피드백, 자동 심사 시스템이 지속적인 품질 신호를 제공하며, 실제 운영 환경의 상호작용에서 얻은 새로운 데이터는 향후 개선을 위해 평가 세트에 통합됩니다.
    - 배포된 에이전트에는 [AI Gateway](https://docs.databricks.com/aws/en/ai-gateway/) 기반의 추론 테이블(inference tables)이 자동으로 활성화됩니다. 이를 통해 DSPy나 LangChain 같은 대중적인 에이전트 저작 라이브러리와 Mosaic AI Agent Framework를 함께 사용할 때 상세한 요청 로그 메타데이터에 접근할 수 있습니다.
    - 엔드투엔드 관측성을 확보하기 위해 [MLflow 추적(tracing))](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/)을 활용할 수 있습니다.
    - 생성형 AI용 [운영 모니터링](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring) 기능을 사용하면 운영 중인 생성형 AI 앱의 추적 데이터에 대해 MLflow 스코어러(scorers)를 자동으로 실행할 수 있습니다.
> 이 강의 이후 데모는 지원되는 프레임워크를 가진 에이전트 프로토타이핑에 초점을 맞출 예정입니다.  

![single-agents-course.png](../Includes/images/single-agents-course.png "single-agents-course.png")
<p>
<em>
이 강의는 에이전트 Framework에 관한 것입니다. 
</em>
</p>

### A1. 프레임워크 아키텍처

노트북, SQL 에디터 및 AI 플레이그라운드에서 유니티 카탈로그(UC)와 외부 도구에 대한 초기 테스트를 마쳤다면, 이제 에이전트 프레임워크를 살펴볼 차례입니다. 이를 지원하기 위해 데이터브릭스(Databricks)는 '모자이크 AI 에이전트 프레임워크(Mosaic AI Agent Framework)'를 통해 고품질의 에이전트 및 RAG 애플리케이션을 개발, 배포, 모니터링할 수 있는 종합 도구를 제공합니다. 이 프레임워크는 다음과 같은 몇 가지 핵심 구성 요소를 중심으로 구축되었습니다.

- **MLflow 3 통합**: 실험 추적, 모델 로깅 및 라이프사이클 관리를 제공합니다.
- **ResponsesAgent 인터페이스**: OpenAI Responses 스키마와 호환되는 프로덕션 환경용 인터페이스입니다.
- **에이전트 개발 라이브러리**: 랭체인(LangChain), 랭그래프(LangGraph), DSPy 및 OpenAI를 지원합니다.
- **Databricks AI 브리지**: 에이전트를 데이터브릭스의 AI 기능과 연결해 주는 연동 패키지입니다.
- **에이전트 거버넌스**: 도구와 에이전트는 유니티 카탈로그(UC)에 등록되어 관리되며, 추론 로그 및 트레이스는 AI 게이트웨이와 MLflow 트레이싱을 통해 관리됩니다. 
- **Mosaic AI Model Serving**: 프로덕션 에이전트를 위한 확장 가능한 배포 인프라를 제공합니다.
- **평가 및 모니터링**: 에이전트의 품질 평가 및 성능 추적을 위한 기본 내장 도구를 포함합니다.

### A2. 요구사항 및 설정

Databricks Framework를 사용해 에이전트를 개발하려면 기술적인 플랫폼 요구사항과 패키지를 알아야 합니다. 

**핵심 요구사항:**
- `databricks-agents` 1.2.0 이상
- `mlflow` 3.1.3 이상
- Python 3.10 이상
- 서버리스 compute 또는 Databricks Runtime 13.3 LTS 이상

**설치 명령:**
```Python
%pip install -U -qqqq databricks-agents mlflow
```

**AI 브리지 통합 패키지:**
Databricks AI 브리지 라이브러리는 Databricks AI/BI Genie, AI search와 같은 Databricks AI 기능과 상호작용할 수 있는 APIs의 공유 계층을 제공합니다. 최신 릴리스 노트와 버전은 [PyPi](https://pypi.org/project/databricks-ai-bridge/)에서 확인할 수 있습니다.
- OpenAI 통합을 위한 `databricks-openai`
- LangChain/LangGraph 통합을 위한 `databricks-langchain`
- DSPy 통합을 위한 `databricks-dspy`
- 전용 통합 패키지 없이 순수 Python 에이전트를 위한 `databricks-ai-bridge`

## B. ResponsesAgent: 프로덕션 인터페이스

Databricks는 운영 등급 에이전트를 생성하는 주요 방법으로 MLflow `ResponsesAgent` 인터페이스를 권장합니다. 이 인터페이스는 OpenAI 응답 스키마와의 호환성을 제공하며 Databricks 특유의 향상된 기능을 추가합니다.

>  기존의 [`ChatAgent`](https://docs.databricks.com/aws/en/generative-ai/agent-framework/agent-legacy-schema)에 익숙하시다면, ResponsesAgent는 새로운 에이전트 개발 시 기존 인터페이스를 대체하기 위해 도입된 기능으로 이해하시면 됩니다.

### B1. ResponsesAgent의 장점

`ResponsesAgent` 인터페이스는 기존의 에이전트 인터페이스와 비교해 상당한 이점을 제공하는 동시에, 지원 프레임워크 내에서 [기존 에이전트를 래핑(Wrapping)할 수 있도록 지원합니다.](https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent?language=Streaming+with+code+re-use#what-if-i-already-have-an-agent). 

**고급 에이전트 기능**
- 복잡한 워크플로우를 위한 멀티 에이전트 지원
- 실시간 응답 청크(chunk)를 활용한 스트리밍 출력
- 포괄적인 툴 호출(Tool-calling) 메시지 이력 관리
- 툴 호출 확인(Confirmation) 기능 지원
- 장시간 실행되는 툴 실행 기능 지원

**개발 및 배포 간소화**
- 프레임워크 독립성: Databricks와 호환되도록 기존의 어떤 에이전트든 래핑
- IDE 자동 완성을 지원하는 타입 기반(Typed) 작성 인터페이스
- 모델 로깅 중 자동 시그니처 추론
    > 권장 `ResponsesAgent` 인터페이스를 사용하지 않는다면, 시그니처를 수동으로 정의하거나 MLflow의 모델 시그니처 추론 기능을 사용하여 입력 예시를 기반으로 에이전트의 시그니처를 자동으로 생성해야 합니다. 자세한 내용은 [여기](https://docs.databricks.com/aws/en/generative-ai/agent-framework/log-agent#infer-model-signature-during-logging)를 참조하세요. 
- `predict` 및 `predict_stream`을 통한 스트리밍 응답 집계 및 자동 추적(Tracing)
    > `ResponsesAgent` 스트리밍이 아닌 요청을 처리하기 위해 `predict`를 반환하는 메서드 `ResponsesAgentResponse`를 구현해야 합니다. 반면, 스트리밍 에이전트의 경우에는 메서드를 `predict_stream` 구현할 수 있습니다. 이 내용은 이 강의의 범위를 넘어서지만, 더 많은 내용은 [여기](https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent?language=Streaming+with+code+re-use)에서 읽어보실 수 있습니다.  
- AI Gateway의 강화된 추론 테이블(Inference Tables)을 통한 상세 로깅

### B2. `ResponsesAgent` 스키마 구조

`ResponsesAgent`는 입력과 출력에 대해 구조화된 스키마를 사용합니다:

**입력 형식 (`ResponsesAgentRequest`):**
```python
{
"input": [
{
"role": "user",
"content": "What did the data scientist say when their Spark job finally completed?"
}
]
}
```

**출력 형식 (`ResponsesAgentResponse`):**
```python
ResponsesAgentResponse(
output=[
{
"type": "message",
"id": str(uuid.uuid4()),
"content": [{"type": "output_text", "text": "Well, that really sparked joy!"}],
"role": "assistant",
}
]
)
```

### B3. 기존 에이전트 래핑

이미 LangChain, LangGraph 또는 이와 유사한 프레임워크로 구축한 에이전트가 있다면, 코드를 처음부터 다시 작성할 필요가 없습니다. 대신 `mlflow.pyfunc.ResponsesAgent`를 상속받는 래퍼(wrapper) 클래스를 생성하면 됩니다.

**기본 래퍼 패턴:**
```python
from uuid import uuid4
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

class MyWrappedAgent(ResponsesAgent):
    def __init__(self, agent):
        # 기존 에이전트(LangChain/LangGraph/OpenAI 등)를 참조하세요.
        self.agent = agent

    def prep_msgs_for_llm(self, messages: list[dict]) -> list[dict]:
        # ResponsesAgentRequest 메시지에서 에이전트가 예상하는 형식으로 변환을 구현하세요
        return messages

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        # 수신 메시지를 에이전트 형식으로 변환하세요
        messages = self.prep_msgs_for_llm([i.model_dump() for i in request.input])

        # 기존 에이전트를 호출하세요 (스트리밍 비중개)
        agent_response = self.agent.invoke(messages)

        # 문자열 출력을 보장하세요; 필요하다면 변환하세요
        if not isinstance(agent_response, str):
            agent_response = str(agent_response)

        # ResponsesAgent 형식으로 변환하기
        output_item = self.create_text_output_item(text=agent_response, id=str(uuid4()))
        return ResponsesAgentResponse(output=[output_item])
```

## C. 지원되는 에이전트 개발 프레임워크

Databricks는 에이전트 개발을 위해 대중적으로 사용되는 다양한 프레임워크를 지원하며, 각 프레임워크는 고유한 강점과 활용 사례를 가지고 있습니다.

### C1. LangChain 통합

LangChain은 광범위한 통합 기능과 역량을 갖추고 있어 LLM 애플리케이션을 구축하는 데 유용한 종합 프레임워크입니다.

**Databricks의 주요 기능**
- Databricks 제공 모델을 LLM 또는 임베딩으로 활용
- Databricks AI Search과의 통합
- MLflow 실험 추적 및 성능 모니터링
- 개발 및 운영 환경의 관측성을 위한 MLflow Tracing 지원
- 원활한 데이터 통합을 위한 PySpark DataFrame 로더
- 자연어 질의를 위한 Spark DataFrame 에이전트 및 Databricks SQL 에이전트 지원

**예시 사용법:**
```python
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
endpoint="databricks-gpt-5-1",
temperature=0.1,
max_tokens=250,
)
chat_model.invoke("How to use Databricks?")
```
> LLM 개발을 위한 [LangChain on Databricks](https://docs.databricks.com/aws/en/large-language-models/langchain)은 현재 실험적 기능 단계이며, 향후 API 정의가 변경될 수 있음에 유의하시기 바랍니다.

### C2. DSPy 프레임워크

[DSPy](https://docs.databricks.com/aws/en/generative-ai/dspy#what-is-dspy)는 자동화된 프롬프트 엔지니어링 기능을 통해 생성형 AI 에이전트를 프로그래밍 방식으로 정의하고 최적화할 수 있는 프레임워크입니다.

**핵심 DSPy 컴포넌트:**
- **모듈(Modules)**: 특정 텍스트 변환을 처리하는 구성 요소(수동으로 작성하던 프롬프트를 대체)
- **시그니처(Signatures)**: 입력과 출력의 동작을 자연어로 서술한 것 ("질문 -> 답변")
- **컴파일러(Compiler)**: 성능 지표에 맞춰 모듈을 조정함으로써 파이프라인을 개선하는 최적화 도구
- **프로그램(Program)**: 복잡한 작업을 수행하기 위해 여러 모듈을 연결한 파이프라인

**DSPy 장점:**
- 자동 프롬프트 최적화
- 에이전트 개선을 위한 체계적 접근법
- 내장된 성능 최적화 기능
- 수동 작업이 필요 없는 프로그래밍 방식의 프롬프트 엔지니어링

### C3. OpenAI 통합

Databricks는 Databricks에서 호스팅하는 모델을 활용하면서도 OpenAI 스타일의 에이전트를 기본적으로 지원합니다.

**통합 이점:**
- 익숙한 OpenAI API 패턴 사용
- Databricks 파운데이션 모델(Foundation Model) API 활용
- OpenAI 모델에서 Databricks 모델로의 원활한 마이그레이션
- 스트리밍 응답과 비스트리밍 응답 모두에 대한 지원
- Databricks 모델을 이용한 함수 호출(Tool-calling) 기능 제공

### C4. 복잡한 워크플로우를 위한 LangGraph

LangGraph는 더 복잡하고 상태 유지가 필요한 워크플로우를 구현할 수 있도록, 그래프 기반의 에이전트 오케스트레이션(조율) 기능을 통해 LangChain의 기능을 확장합니다.

**LangGraph 기능:**
- 그래프 기반의 에이전트 워크플로우
- 에이전트 상호작용 전반의 상태 관리
- 복잡한 의사결정 트리 및 조건부 로직
- 다단계 추론 및 도구 연동 조율

## D. 스트리밍 및 실시간 응답

스트리밍 기능을 통해 에이전트는 실시간 응답을 여러 조각(chunk)으로 나누어 제공할 수 있으며, 이를 통해 사용자 경험을 향상시키고 대화형 애플리케이션을 구현할 수 있습니다. 기존 방식은 사용자에게 결과를 전송하기 전 전체 응답이 완성될 때까지 기다리는 것이었습니다. 반면 MLflow를 사용하면 에이전트의 최종 응답뿐만 아니라, 이러한 응답 조각들과 추론 과정까지 확인할 수 있어 에이전트가 특정 도구를 선택하거나 선택하지 않은 이유에 대한 통찰을 얻을 수 있습니다.

> [Agent Bricks 기반의 지식 어시스턴트(Knowledge Assistant)](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant)는 스트리밍 기능을 완벽히 지원하며, 아래 스크린샷에서 지식 어시스턴트 응답의 트레이스(Trace)를 통해 이를 확인할 수 있습니다.

![mlflow-ui.png](../Includes/images/mlflow-ui.png "mlflow-ui.png")
<p><em>Knowledge Assistant와 Agent Bricks를 통해 MLflow로 청크 트레이싱을 한 예시입니다.</em></p>

### D1. 스트리밍 구현

`ResponsesAgent`를 사용하여 스트리밍을 구현하려면 다음 패턴을 따르십시오.

1. **델타 이벤트(Delta Events) 발행**: 동일한 `item_id`를 가진 여러 `output_text.delta` 이벤트를 전송합니다.
2. **완료 이벤트로 종료**: 전체 출력이 포함된 최종 `response.output_item.done` 이벤트를 전송합니다

**streaming 장점:**
- 실시간 사용자 피드백 제공
- 체감 성능 향상
- 장시간 실행되는 작업에 대한 사용자 참여도 개선
- MLflow 추적(tracing) 자동 연동
- AI Gateway 추론 테이블의 응답 데이터 집계

![mlflow-chunking.png](../Includes/images/mlflow-chunking.png "mlflow-chunking.png")
<p>&lt;em&gt;
Streaming 응답은 완전한 생각으로 표현되는 출력 덩어리를 통해 MLflow와 함께 스트리밍됩니다. 
&lt;/em&gt;
</p>


### D2. Streaming에서의 오류 처리

Mosaic AI는 databricks_output.error 아래에 있는 마지막 토큰을 통해 스트리밍 오류를 전달합니다.

```json
{
  "delta": "...",
  "databricks_output": {
    "trace": {...},
    "error": {
      "error_code": "BAD_REQUEST",
      "message": "TimeoutException: Tool XYZ failed to execute."
    }
  }
}
```

**참고:** 클라이언트 애플리케이션은 이러한 오류를 적절히 처리하고 화면에 표시해야 합니다.

## E. (선택 사항) 고급 기능 및 커스터마이징

Databricks 에이전트 Framework는 정교한 에이전트 구현을 위한 여러 고급 기능을 제공합니다. 아래에 제시된 주제들은 이 강의의 범위를 넘어섭니다.

### E1. 사용자 정의 입력 및 출력

일부 시나리오에서는 대화 기록에 포함되지 않아야 하는 추가적인 에이전트 입력(예: client_type, session_id)이나 출력(예: 정보 검색 출처 링크)이 필요할 수 있습니다.

**사용자 정의 필드 지원:**
- `custom_inputs`: 표준 메시지 외의 추가 입력 매개변수입니다. 사용자 정의 입력 및 출력에 대한 자세한 내용은 [여기](https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent#custom-inputs-and-outputs)에서 확인할 수 있습니다. 
- `custom_outputs`: 대화 흐름의 일부가 아닌 추가 출력 데이터입니다.
- 에이전트 코드 내 `request.custom_inputs`을 통해 접근할 수 있습니다.
- AI Playground 및 리뷰 앱에서 JSON 구성(Configuration)을 지원합니다.

**중요한 제한사항:**
Agent Evaluation 리뷰 앱은 추가 입력 필드가 있는 에이전트의 트레이스(Trace) 시각화를 지원하지 않습니다.

> 이와 관련된 고급 기능에 대해 더 읽어보실 수 있습니다 [여기](https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent#advanced-features). 이 내용은 이 과정의 범위를 넘어섭니다.

### E2. 리트리버 통합 및 스키마

AI 에이전트는 일반적으로 벡터 검색 인덱스의 비정형 데이터에 리트리버(Retriever)를 사용합니다. Databricks는 리트리버 트레이싱 및 평가를 위한 전용 기능을 제공합니다. 리트리버 통합에 대한 자세한 내용은 [여기](https://docs.databricks.com/aws/en/generative-ai/agent-framework/unstructured-retrieval-tools#set-retriever-schema-to-ensure-mlflow-compatibility)에서 확인할 수 있습니다.

**리트리버 장점**
- AI Playground UI에서 소스 문서 링크 자동 생성
- 검색 결과의 근거성(Groundedness) 및 관련성 자동 평가
- Databricks AI Bridge 리트리버 도구와의 통합

**커스텀 리트리버 스키마:**
```python
import mlflow

mlflow.models.set_retriever_schema(
name="mlflow_docs_vector_search",
primary_key="document_id",      # 문서 ID 필드
text_column="chunk_text",       # 내용 필드
doc_uri="doc_uri",             # 문서 URI 필드
other_columns=["title"],        # 추가 메타데이터
)
```

> 이 내용은 이 과정의 범위를 넘어섭니다. 리트리버 도구에 대해 더 읽어보실 수 있습니다 [여기](https://docs.databricks.com/aws/en/generative-ai/agent-framework/unstructured-retrieval-tools).

### E3. 멀티 에이전트 시스템(Multi-Agent Systems)

Databricks는 여러 전문 에이전트가 협력하여 문제를 해결하는 복잡한 다중 에이전트 시스템을 지원합니다.

> Databricks는 여러 전문 에이전트가 협력하여 문제를 해결하는 복잡한 멀티 에이전트 시스템을 지원합니다. 이 내용은 본 과정의 범위를 벗어납니다.[Genie](https://docs.databricks.com/aws/en/generative-ai/agent-framework/multi-agent-genie)와 [Agent Bricks](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/multi-agent-supervisor)에 관한 문서를 읽어보실 수 있습니다. 다중 에이전트 시스템과 함께하는 Genie에 대해 [여기](https://docs.databricks.com/aws/en/generative-ai/agent-framework/) 에서 더 읽어보실 수 있습니다.

### E4. 상태 저장 에이전트 (Stateful Agents)

상태 기반 에이전트는 대화 스레드 전반에 걸쳐 메모리를 유지하고 대화 체크포인트 기능을 제공할 수 있습니다.

> 이 내용은 이 과정의 범위를 넘어섭니다. 상태 에이전트에 대해 더 알고 싶다면 [이 문서](https://docs.databricks.com/aws/en/generative-ai/agent-framework/stateful-agents) 문서를 참고하세요. 상태가 있는 AI 에이전트에 대해 더 읽어보실 수 있습니다 [여기](https://docs.databricks.com/aws/en/generative-ai/agent-framework/stateful-agents).

## 결론

Databricks Mosaic AI 에이전트 프레임워크는 실제 서비스 환경에 바로 적용할 수 있는 AI 에이전트를 구축하기 위한 종합적인 플랫폼을 제공합니다. 핵심 요약으로는 널리 쓰이는 프레임워크(예: LangChain)의 강점 파악, 에이전트 라이프사이클 구축 시의 모범 사례에 대한 이해 및 개발, 그리고 실제 서비스 배포 시 고려사항 등이 있습니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>